# 3-stage: локальная визуализация предсказания

Ноутбук всегда загружает модель из `3-stage/best_model.pth`, выбирает `.npy` КТ-срез и показывает итоговое наложение:

- CT slice — серый фон;
- ground truth mask — синяя область;
- predicted mask — красная область.

U-Net берётся из `2-stage/model.py`. Порог по умолчанию берётся из checkpoint (`best_threshold`).


## 1. Импорты

In [1]:
from pathlib import Path
import csv
import importlib.util
import json
import random
import matplotlib.pyplot as plt
import numpy as np
import torch

REPO_ROOT_HINT = Path.home() / "Coding" / "PycharmProjects" / "Lung-Tumor-Segmentation"


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, REPO_ROOT_HINT]
    for candidate in candidates:
        candidate = candidate.expanduser().resolve(strict=False)
        if (candidate / "2-stage" / "model.py").exists():
            return candidate
    raise FileNotFoundError(
        "Не найден 2-stage/model.py. Запусти notebook из корня репозитория "
        "Lung-Tumor-Segmentation или скопируй папку 2-stage рядом с 3-stage."
    )


REPO_ROOT = find_repo_root()
MODEL_FILE = REPO_ROOT / "2-stage" / "model.py"
spec = importlib.util.spec_from_file_location("lung_unet_model", MODEL_FILE)
if spec is None or spec.loader is None:
    raise ImportError(f"Не удалось загрузить model.py: {MODEL_FILE}")
model_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_module)
build_model = model_module.build_model


print("repo:", REPO_ROOT)
print("model.py:", MODEL_FILE)
print("torch:", torch.__version__)

repo: /Users/daniil/Coding/PycharmProjects/Lung-Tumor-Segmentation
model.py: /Users/daniil/Coding/PycharmProjects/Lung-Tumor-Segmentation/2-stage/model.py
torch: 2.9.1


## 2. Пути

Положи checkpoint строго по пути:

```text
3-stage/best_model.pth
```

Если локально есть `preprocessed_npy/test/...`, можно выбрать срез через `manifest.csv`. Если нет, сначала скачай нужный case с сервера, например `lung_096`.


In [2]:
CHECKPOINT_PATH = REPO_ROOT / "3-stage" / "best_model.pth"
DATA_DIR = REPO_ROOT / "preprocessed_npy"
OUTPUT_DIR = REPO_ROOT / "3-stage" / "predictions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT_PATH}\n"
        "Положи обученную модель в 3-stage/best_model.pth"
    )

print("checkpoint:", CHECKPOINT_PATH)
print("data dir:", DATA_DIR)
print("output dir:", OUTPUT_DIR)


(True,
 True,
 PosixPath('/Users/daniil/Coding/PycharmProjects/Lung-Tumor-Segmentation/3-stage/predictions'))

## 3. Вспомогательные функции

In [3]:
def load_checkpoint(path: Path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def choose_device():
    if torch.cuda.is_available():
        return torch.device("cuda:0")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def as_chw_float32(array: np.ndarray) -> np.ndarray:
    array = np.asarray(array)
    if array.ndim == 2:
        array = array[None, :, :]
    elif array.ndim == 3:
        if array.shape[0] == 1:
            pass
        elif array.shape[-1] == 1:
            array = np.moveaxis(array, -1, 0)
        else:
            raise ValueError(f"Unsupported array shape: {array.shape}")
    else:
        raise ValueError(f"Unsupported array shape: {array.shape}")
    return array.astype(np.float32, copy=False)


def masked(mask: np.ndarray):
    return np.ma.masked_where(mask == 0, mask)


def resolve_manifest_path(raw_path: str, data_dir: Path, split: str, case_id: str, kind: str) -> Path:
    raw = Path(raw_path)
    folder = "images" if kind == "image" else "masks"
    candidates = []
    if raw.is_absolute():
        candidates.append(raw)

    for marker in (data_dir.name, "preprocessed_npy"):
        if marker in raw.parts:
            idx = raw.parts.index(marker)
            if idx + 1 < len(raw.parts):
                candidates.append(data_dir / Path(*raw.parts[idx + 1:]))

    candidates += [
        data_dir / split / case_id / folder / raw.name,
        data_dir / raw,
        REPO_ROOT / raw,
    ]

    for candidate in candidates:
        candidate = candidate.expanduser().resolve(strict=False)
        if candidate.exists():
            return candidate
    return candidates[0].expanduser().resolve(strict=False)


def require_existing_file(path: Path, label: str) -> Path:
    path = Path(path).expanduser().resolve(strict=False)
    if path.exists():
        return path
    raise FileNotFoundError(
        f"{label} file not found: {path}\n\n"
        "В локальной папке preprocessed_npy должны быть не только manifest.csv/splits.json, "
        "но и сами .npy файлы.\n"
        "Для текущего примера скачай case lung_096 с сервера:\n\n"
        "cd /Users/daniil/Coding/PycharmProjects/Lung-Tumor-Segmentation\n"
        "mkdir -p preprocessed_npy/test\n"
        "rsync -avh --progress --partial -e 'ssh -p 48806' "
        "user@195.208.16.1:~/Lung-Tumor-Segmentation/preprocessed_npy/test/lung_096/ "
        "./preprocessed_npy/test/lung_096/"
    )


def pick_manifest_row(data_dir: Path, split="test", sample="first-positive", case_id=None, z=None):
    manifest_path = data_dir / "manifest.csv"
    with manifest_path.open("r", newline="", encoding="utf-8") as f:
        rows = [row for row in csv.DictReader(f) if row["split"] == split]

    if case_id is not None:
        rows = [row for row in rows if row["case_id"] == case_id]
    if z is not None:
        rows = [row for row in rows if int(float(row["z"])) == int(z)]
    if sample in {"first-positive", "random-positive"}:
        rows = [row for row in rows if int(float(row["has_tumor"])) == 1]
    if not rows:
        raise ValueError("No matching manifest rows")
    return random.choice(rows) if sample.startswith("random") else rows[0]


def compute_slice_metrics(pred_mask: np.ndarray, gt_mask: np.ndarray | None):
    if gt_mask is None:
        return None
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)
    tp = float(np.logical_and(pred, gt).sum())
    fp = float(np.logical_and(pred, ~gt).sum())
    fn = float(np.logical_and(~pred, gt).sum())
    eps = 1e-7
    return {
        "dice": (2.0 * tp) / (2.0 * tp + fp + fn + eps),
        "precision": tp / (tp + fp + eps),
        "recall": tp / (tp + fn + eps),
        "pred_pixels": float(pred.sum()),
        "gt_pixels": float(gt.sum()),
    }

## 4. Загрузка модели

In [4]:
checkpoint = load_checkpoint(CHECKPOINT_PATH)
config = checkpoint.get("config", {})
img_size = int(config.get("data", {}).get("img_size", 512))
threshold = float(checkpoint.get("best_threshold", config.get("metrics", {}).get("threshold", 0.5)))

device = choose_device()
model = build_model(config.get("model", {})).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("device:", device)
print("img_size:", img_size)
print("threshold:", threshold)
print("checkpoint epoch:", checkpoint.get("epoch"))
print("best_val_dice:", checkpoint.get("best_val_dice"))

device: mps
img_size: 512
threshold: 0.7
checkpoint epoch: 49
best_val_dice: 0.45435737495093276


## 5. Выбор среза

Вариант A: через `manifest.csv`, если локально есть `.npy` файлы.

По умолчанию берётся первый positive test slice. Можно указать `CASE_ID` и `Z` вручную.

In [ ]:
SPLIT = "test"
SAMPLE = "first-positive"  # first-positive | random-positive | first | random
CASE_ID = "lung_096"             # например "lung_096"
Z = 92                  # например 92

row = pick_manifest_row(DATA_DIR, split=SPLIT, sample=SAMPLE, case_id=CASE_ID, z=Z)
case_id = row["case_id"]
z = int(float(row["z"]))

image_path = resolve_manifest_path(row["image_path"], DATA_DIR, row["split"], case_id, "image")
mask_path = resolve_manifest_path(row["mask_path"], DATA_DIR, row["split"], case_id, "mask")
prefix = f"{row['split']}_{case_id}_z{z:04d}"

print("image:", image_path)
print("mask:", mask_path)
print("exists:", image_path.exists(), mask_path.exists())

Вариант B: если хочешь указать файл вручную, раскомментируй и запусти эту ячейку вместо предыдущей.

In [ ]:
# image_path = REPO_ROOT / "preprocessed_npy/test/lung_096/images/z0092.npy"
# mask_path = REPO_ROOT / "preprocessed_npy/test/lung_096/masks/z0092.npy"
# prefix = "lung_096_z0092"
# print(image_path, image_path.exists())
# print(mask_path, mask_path.exists())

## 6. Inference

In [ ]:
image_path = require_existing_file(image_path, "image")
mask_path = require_existing_file(mask_path, "mask") if mask_path is not None else None

image_chw = as_chw_float32(np.load(image_path, allow_pickle=False))
if tuple(image_chw.shape) != (1, img_size, img_size):
    raise ValueError(f"Image shape {image_chw.shape}, expected {(1, img_size, img_size)}")

gt_mask = None
if mask_path is not None and Path(mask_path).exists():
    gt_mask = (as_chw_float32(np.load(mask_path, allow_pickle=False))[0] > 0.5).astype(np.uint8)

x = torch.from_numpy(np.ascontiguousarray(image_chw)).unsqueeze(0).to(device)
with torch.no_grad():
    logits = model(x)
    probability = torch.sigmoid(logits)[0, 0].detach().cpu().numpy().astype(np.float32)

pred_mask = (probability >= threshold).astype(np.uint8)
metrics = compute_slice_metrics(pred_mask, gt_mask)

print("probability range:", float(probability.min()), float(probability.max()))
print("pred pixels:", int(pred_mask.sum()))
if metrics:
    print(metrics)

## 7. Визуализация

На итоговом изображении:

- синяя область — настоящая маска опухоли (`ground truth`);
- красная область — предсказанная маска модели (`prediction`).


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(image_chw[0], cmap="gray", vmin=0, vmax=1)

if gt_mask is not None:
    ax.imshow(masked(gt_mask), cmap="Blues", alpha=0.55, vmin=0, vmax=1)
ax.imshow(masked(pred_mask), cmap="Reds", alpha=0.55, vmin=0, vmax=1)

title = f"CT overlay | threshold={threshold:.3f}"
if metrics:
    title += f"\nDice={metrics['dice']:.3f}, Precision={metrics['precision']:.3f}, Recall={metrics['recall']:.3f}"
ax.set_title(title)
ax.axis("off")

from matplotlib.patches import Patch
legend_items = []
if gt_mask is not None:
    legend_items.append(Patch(facecolor="tab:blue", alpha=0.55, label="Ground truth"))
legend_items.append(Patch(facecolor="tab:red", alpha=0.55, label="Prediction"))
ax.legend(handles=legend_items, loc="lower right")

fig.tight_layout()
plt.show()


## 8. Сохранение результата

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
png_path = OUTPUT_DIR / f"{prefix}_prediction.png"
pred_mask_path = OUTPUT_DIR / f"{prefix}_pred_mask.npy"
pred_prob_path = OUTPUT_DIR / f"{prefix}_pred_prob.npy"
summary_path = OUTPUT_DIR / f"{prefix}_summary.json"

np.save(pred_mask_path, pred_mask)
np.save(pred_prob_path, probability)

fig.savefig(png_path, dpi=170, bbox_inches="tight")

summary = {
    "checkpoint": str(CHECKPOINT_PATH),
    "image": str(image_path),
    "mask": str(mask_path) if mask_path is not None else None,
    "threshold": threshold,
    "device": str(device),
    "metrics": metrics,
    "outputs": {
        "png": str(png_path),
        "pred_mask": str(pred_mask_path),
        "pred_prob": str(pred_prob_path),
    },
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("saved:", png_path)
print("saved:", pred_mask_path)
print("saved:", pred_prob_path)
print("saved:", summary_path)